# Sprint 2 · Protocolo de validación y verdad de referencia (plata / oro)

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

Fija **cómo se evaluará** el motor de causa raíz antes de construirlo (`docs/04_protocolo_validacion.md`):
la unidad de evaluación (**incidente** sobre el grafo), la verdad de referencia en dos niveles (**plata** = manuales Huawei,
**oro** = NOC), las particiones temporales, el enmascaramiento y las métricas. Código en `src/etiquetado.py`,
`src/evaluacion.py` y `src/sintetico.py`.

**Contenido**
1. Datos y grafo · 2. Incidentes sobre el grafo · 3. Un incidente, interactivo · 4. Particiones temporales ·
5. Enmascaramiento · 6. Etiqueta oro: planilla para el NOC · 7. Etiqueta plata: reglas de los manuales ·
8. Métodos de referencia en la red real · 9. Prueba de humo con avalanchas sintéticas · 10. Conclusiones


## 1. Datos y grafo

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.ingesta import construir_dataset
from src.grafo import construir_grafo, grafo_sitios, jerarquia_sitios
from src.etiquetado import (construir_incidentes, seleccionar_muestra_oro, planilla_oro, cargar_oro,
                            cargar_reglas, etiqueta_plata, acuerdo_plata_oro)
from src.manuales import revision_a_reglas
from src.evaluacion import (folds_temporales, tabla_folds, enmascarar, rankear, comparar, evaluar,
                            evaluar_por_fold, mcnemar, verdad_desde_simulacion)
from src.sintetico import simular_avalanchas
from src.visual import vista_incidente, mostrar, guardar_html

BASE_DIR = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed" / "dataset_consolidado.parquet"
REPORTES = ROOT / "reports" / "sprint2"
REGLAS = ROOT / "docs" / "reglas_causales_huawei.csv"
ORO_COMPLETADA = ROOT / "data" / "interim" / "oro_noc_completada.xlsx"   # la planilla que devuelve el NOC

df = pd.read_parquet(PROC) if PROC.exists() else construir_dataset(str(BASE_DIR))
G = construir_grafo(str(BASE_DIR / "red.xlsx"), verbose=False)
J = jerarquia_sitios(G, grafo_sitios(G))
print(f"{len(df):,} alarmas · {df['ventana_id'].nunique()} ventanas · grafo con {G.number_of_nodes():,} nodos")

> **Resumen.** Carga el dataset consolidado del Sprint 1 (o lo reconstruye desde `data/raw`) y el grafo del notebook 04. La jerarquía `J` (saltos de cada sitio al NOC) se usa para desempatar por topología.

## 2. Incidentes sobre el grafo

**Incidente** = alarmas **nuevas** de una misma ventana en equipos conectados en el grafo físico a ≤ 4 saltos, con al
menos 3 alarmas. Es la unidad que se etiqueta y se evalúa (una ventana puede tener varias fallas independientes).

**Hora de cada alarma.** El NCE guarda varias marcas de tiempo; en alarmas crónicas, la de «ocurrencia» puede ser de hace
semanas. Se usa la primera marca que cae **dentro de la ventana** (`t_evento`); si ninguna cae, la alarma es
**reaparecida** (no inicia una falla ahora) y no entra al incidente. La hora del nombre de la carpeta (local) se pasa a UTC
con el desfase detectado automáticamente (−5 h para Lima).

In [ ]:
df_inc, incidentes = construir_incidentes(df, G, max_saltos=4, min_alarmas=3, verbose=True)
print(f"Incidentes: {len(incidentes):,} · alarmas en incidentes: {len(df_inc):,} "
      f"({len(df_inc)/max(len(df),1):.1%} del total)")
display(incidentes.sort_values("n_alarmas", ascending=False).head(10)[
    ["incidente_id", "ventana_ts", "red", "n_alarmas", "n_equipos", "n_sitios", "primeras_alarmas"]])

> **Qué muestra la tabla.** Los 10 incidentes más grandes: cuándo ocurrieron, en qué red, cuántas alarmas y equipos agrupan y las tres primeras alarmas en orden de ocurrencia. Que la mayoría de alarmas **no** quede en incidentes es esperable: muchas son crónicas (siguen activas de ventanas anteriores) o aisladas.

In [ ]:
if len(incidentes):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    incidentes["n_alarmas"].clip(upper=40).plot.hist(bins=20, ax=ax[0], color="#0B5D8F")
    ax[0].set(title="Tamaño de los incidentes", xlabel="alarmas por incidente (40 = 40 o más)", ylabel="incidentes")
    incidentes["red"].value_counts().plot.barh(ax=ax[1], color="#2BA88A")
    ax[1].set(title="Incidentes por red", xlabel="incidentes")
    plt.tight_layout(); plt.show()
else:
    print("No hay incidentes con los datos actuales.")

> **Qué muestran los gráficos.** *(Izq.)* Distribución del tamaño de los incidentes: sirve para ajustar el mínimo de 3 alarmas y el radio de 4 saltos. *(Der.)* Cuántos incidentes son solo de acceso (`ax`), solo de transporte (`tx`) o cruzan ambas redes (`ax+tx`); estos últimos son los más difíciles de diagnosticar a mano.

## 3. Un incidente, interactivo

El incidente más grande dibujado sobre el grafo físico: arriba lo más cercano al NOC. Pasa el cursor sobre un equipo para
ver sus alarmas en orden de llegada. La etiqueta `#k` es la posición que le da el método topológico (sección 8).

In [ ]:
if len(incidentes):
    INC = incidentes.sort_values("n_alarmas", ascending=False)["incidente_id"].iloc[0]
    alarmas_inc = df_inc[df_inc["incidente_id"] == INC]
    rk = rankear(alarmas_inc, "topologico", G=G)
    html = vista_incidente(G, alarmas_inc, ranking=rk, J=J, titulo=f"Incidente {INC}")
    mostrar(html, alto=620)
    guardar_html(html, REPORTES / "incidente_mayor.html")

> **Qué muestra.** Los equipos con alarmas (rojo), los equipos y enlaces intermedios que los conectan (gris) y el orden del método topológico. Es la vista que verá el operador del NOC al validar el diagnóstico.

## 4. Particiones temporales (origen móvil)

Los cortes se hacen por **incidentes**, no por ventanas: el primer 40 % de los incidentes (en orden de tiempo) solo
entrena y el resto se divide en 4 bloques consecutivos con una cantidad parecida de incidentes. Cada fold entrena con todo
lo anterior y deja 1 ventana de margen. Nunca se mezclan ventanas al azar (el modelo no ve el futuro).

In [ ]:
folds = folds_temporales(df[["ventana_id", "ventana_ts"]], n_folds=4, min_entrenamiento=0.4, separacion=1,
                        pesos=incidentes.groupby("ventana_id").size())
tf = tabla_folds(folds)
tf["incidentes_test"] = [incidentes["ventana_id"].isin(f["test"]).sum() for f in folds]
display(tf)

> **Qué muestra la tabla.** Para cada fold: cuántas ventanas entrenan, cuántas se prueban, el período de prueba y cuántos incidentes caen en él. Con pocas semanas de datos, los folds son cortos; se recalcula al ampliar la descarga.

In [ ]:
if len(tf):
    vts = df[["ventana_id", "ventana_ts"]].drop_duplicates().set_index("ventana_id")["ventana_ts"]
    fig, ax = plt.subplots(figsize=(11, 2.2))
    for i, f in enumerate(folds):
        tr, te = vts.loc[f["train"]], vts.loc[f["test"]]
        if len(tr):
            ax.plot([tr.min(), tr.max()], [i, i], lw=8, color="#9FB0C0", solid_capstyle="butt")
        ax.plot([te.min(), te.max()], [i, i], lw=8, color="#D1495B", solid_capstyle="butt")
    ax.set_yticks(range(len(folds)), [f"fold {f['fold']}" for f in folds]); ax.invert_yaxis()
    ax.set_title("Entrenamiento (gris) y prueba (rojo) por fold"); plt.tight_layout(); plt.show()

> **Qué muestra el gráfico.** La línea de tiempo de cada fold: el entrenamiento (gris) crece y la prueba (rojo) siempre queda **después**. Así se simula el uso real: diagnosticar fallas nuevas con lo aprendido del pasado.

## 5. Enmascaramiento (anti-fuga)

In [ ]:
df_mask, quitadas = enmascarar(df)
print("Columnas quitadas:", quitadas or "ninguna (el export actual del NCE no trae campos de reconocimiento ni de alarma raíz)")

> **Resumen.** Antes de entregar los datos a un método se quitan las columnas que delatarían la respuesta (reconocimiento del operador, comentarios, banderas de alarma raíz del gestor, hora de cierre). El filtro queda activo por si cambian las descargas.

## 6. Etiqueta **oro**: planilla para el NOC

Si hay 100 incidentes o menos, se etiquetan **todos** (mejor que una muestra). Si hay más: muestra reproducible
(semilla 42) de los 30 más grandes + 30 al azar estratificados por red. La planilla tiene
instrucciones, una fila por incidente con columnas amarillas para completar y la hoja de alarmas para decidir.

In [ ]:
if len(incidentes):
    muestra = seleccionar_muestra_oro(incidentes, n_grandes=30, n_aleatorios=30, todos_si_menos_de=100)
    ruta = planilla_oro(df_inc, muestra, REPORTES / "planilla_oro_NOC.xlsx")
    print(f"Planilla generada: {ruta}  ({len(muestra)} incidentes)")
    display(muestra.groupby(["motivo", "red"]).size().rename("incidentes").reset_index())

> **Qué muestra la tabla.** Cuántos incidentes de la muestra salen por tamaño (`grande`) y al azar (`aleatorio`), por red. El archivo `reports/sprint2/planilla_oro_NOC.xlsx` se envía al NOC; la devuelta se guarda como `data/interim/oro_noc_completada.xlsx`.

In [ ]:
oro = cargar_oro(ORO_COMPLETADA) if ORO_COMPLETADA.exists() else None
print(f"Etiquetas oro cargadas: {len(oro)}" if oro is not None else "Aún no hay etiquetas oro (pendiente del NOC).")

## 7. Etiqueta **plata**: reglas de los manuales Huawei

`docs/reglas_causales_huawei.csv` reúne las reglas causa→efecto de los manuales (con fuente y confianza). En cada
incidente, la causa plata es la alarma que explica a otras y no es explicada por ninguna; si hay varias, gana la más
cercana al NOC y luego la más temprana.

In [ ]:
# Si ya revisaste la planilla de reglas (columna «decision»), se cargan las aceptadas como versión v1-RTN320.
REVISION = REPORTES / "revision_reglas_RTN320.xlsx"
if REVISION.exists() and pd.read_excel(REVISION, sheet_name="Candidatos")["decision"].notna().any():
    revision_a_reglas(REVISION, REGLAS, version="v1-RTN320")
reglas = cargar_reglas(REGLAS)
print(f"Reglas cargadas: {len(reglas)}")
if len(reglas) and len(incidentes):
    plata = etiqueta_plata(df_inc, reglas, G)
    display(plata["plata_estado"].value_counts().rename("incidentes").to_frame())
    if oro is not None:
        print("Acuerdo plata–oro:", acuerdo_plata_oro(plata, oro))
else:
    plata = None
    print("Pendiente: revisar reports/sprint2/revision_reglas_RTN320.xlsx y cargarla con revision_a_reglas.")
if len(reglas):
    display(reglas.groupby(["version", "relacion"]).size().rename("reglas").reset_index())
if plata is not None:
    por_red = incidentes[["incidente_id", "red"]].merge(plata, on="incidente_id")
    display(pd.crosstab(por_red["red"], por_red["plata_estado"], margins=True, margins_name="Total"))

> **Resumen.** Cuando se carguen las reglas, la tabla dirá cuántos incidentes tienen causa plata única, cuántos requirieron desempate topológico y cuántos no tienen ninguna regla aplicable (cobertura de los manuales). La tabla por red muestra dónde hay plata: con solo el manual del RTN 320 (v1) se espera cobertura en `ax` y poca en `tx`; los resultados con plata se reportan **por red** hasta cargar los manuales de routers (v2).

### 7.1 Cobertura de los manuales y prioridad de revisión

¿Qué alarmas conviene buscar primero en los manuales de routers? Las que concentran más volumen en los incidentes y
aún no están en ningún catálogo. La tabla ordena los tipos de alarma por cantidad, con el % acumulado.

In [ ]:
from src.manuales import cobertura_alarmas
CATALOGO = ROOT / "docs" / "catalogo_alarmas.csv"
catalogo = pd.read_csv(CATALOGO) if CATALOGO.exists() else pd.DataFrame(columns=["alarma"])
base = df_inc if len(df_inc) else df
cob = cobertura_alarmas(base, catalogos=[catalogo], reglas=reglas)
print(f"Tipos de alarma: {len(cob)} · en algún manual: {cob['en_manual'].sum()} "
      f"({cob.loc[cob['en_manual'], 'alarmas'].sum() / max(cob['alarmas'].sum(), 1):.1%} del volumen)")
faltan = cob[~cob["en_manual"]]
print(f"Para cubrir el 90 % del volumen sin manual basta revisar {int((faltan['alarmas'].cumsum() / max(faltan['alarmas'].sum(), 1) < 0.9).sum() + 1) if len(faltan) else 0} tipos.")
display(cob.head(25))
cob.to_csv(REPORTES / "cobertura_alarmas.csv", index=False)

> **Qué muestra la tabla.** Los 25 tipos de alarma con más volumen (en los incidentes): cuántas alarmas, su % y % acumulado, si ya están en el catálogo de un manual (`en_manual`), si ya tienen regla (`con_regla`) y en qué rol de equipo aparecen. Las filas con `en_manual = False` y rol de router son la **lista corta** para buscar en los manuales del ATN 910C y NE40E. El archivo completo queda en `reports/sprint2/cobertura_alarmas.csv`.

## 8. Métodos de referencia en la red real

T0 = el operador ordena por hora · T0′ = la más severa primero · T1 = la más cercana al NOC primero. Sin verdad todavía
no hay aciertos que medir; se muestra **cuánto coinciden** entre sí (si coinciden siempre, el problema sería trivial).

In [ ]:
if len(incidentes):
    rk_real = {m: rankear(df_inc, m, G=G) for m in ["temporal", "severidad", "topologico"]}
    top1 = pd.DataFrame({m: r[r["rank"] == 1].set_index("incidente_id")["equipo"] for m, r in rk_real.items()})
    print(f"Incidentes donde los 3 métodos proponen el mismo equipo: {(top1.nunique(axis=1) == 1).mean():.1%}")
    verdad_real = oro if oro is not None else None
    if verdad_real is not None:
        display(comparar(rk_real, verdad_real))

> **Resumen.** Un porcentaje bajo de coincidencia indica que la elección de causa raíz **no es obvia** y justifica un motor que combine tiempo y topología. Cuando lleguen las etiquetas oro, la misma celda muestra la tabla de métricas.

## 9. Prueba de humo con avalanchas sintéticas

Mientras llegan las etiquetas, se inyectan fallas **conocidas** sobre la topología real (`src/sintetico.py`): falla un
equipo, los de aguas abajo emiten alarmas con retardo y *jitter*, a veces la raíz llega tarde y hay ruido. Verifica que el
protocolo funciona de punta a punta. **No** mide el desempeño en la red real (los tipos de alarma son `SIM_…`).

In [ ]:
df_sim, verdad_sim, reglas_sim = simular_avalanchas(G, n_ventanas=400, semilla=42)
inc_df_sim, inc_sim = construir_incidentes(df_sim, G)
V = verdad_desde_simulacion(inc_df_sim, verdad_sim)
rk_sim = {"T0 · temporal": rankear(inc_df_sim, "temporal"),
          "T0′ · severidad": rankear(inc_df_sim, "severidad"),
          "T1 · topológico": rankear(inc_df_sim, "topologico", G=G)}
tabla = comparar(rk_sim, V)
print(f"{len(inc_sim)} incidentes sintéticos · {len(V)} con la causa inyectada dentro del incidente")
display(tabla)

> **Qué muestra la tabla.** Para cada método: Top-1/3/5, MRR, precisión, cobertura (recall), F1, equipos que el operador revisaría hasta dar con la causa (proxy del tiempo de localización), tiempo de cómputo e IC 95 % del Top-1. Ningún método simple acierta siempre: es el margen que debe cubrir el motor del Sprint 3.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
x = range(len(tabla)); w = 0.25
for j, k in enumerate(["top_1", "top_3", "top_5"]):
    ax.bar([i + (j - 1) * w for i in x], tabla[k], width=w, label=k.replace("_", "-").title(),
           color=["#0B5D8F", "#2BA88A", "#F2A541"][j])
ax.set_xticks(list(x), tabla["metodo"]); ax.set_ylim(0, 1); ax.set_ylabel("exactitud")
ax.set_title("Métodos de referencia sobre avalanchas sintéticas"); ax.legend(frameon=False, ncol=3)
plt.tight_layout(); plt.show()

> **Qué muestra el gráfico.** La exactitud Top-1, Top-3 y Top-5 de cada método de referencia. La distancia entre Top-1 y Top-3 indica que la causa suele estar entre los primeros candidatos pero mal ordenada.

In [ ]:
_, d_t = evaluar(rk_sim["T0 · temporal"], V)
_, d_g = evaluar(rk_sim["T1 · topológico"], V)
b, c_, p = mcnemar(d_t, d_g)
print(f"McNemar T0 vs T1: solo acierta T0 = {b}, solo acierta T1 = {c_}, p = {p}")
folds_sim = folds_temporales(inc_df_sim[["ventana_id", "ventana_ts"]], n_folds=4)
display(evaluar_por_fold(inc_df_sim, inc_sim, V, folds_sim, ["temporal", "topologico"], G=G)
        .groupby("metodo")[["top_1", "top_3", "mrr"]].agg(["mean", "std"]).round(3))

> **Qué muestra.** *(Arriba)* La prueba de McNemar compara dos métodos sobre los **mismos** incidentes: un p < 0,05 indica que la diferencia no es casualidad. *(Abajo)* La media y desviación de las métricas a lo largo de los 4 folds: una desviación pequeña indica que el resultado es estable en el tiempo.

In [ ]:
pl_sim = etiqueta_plata(inc_df_sim, reglas_sim, G)
print("Plata (reglas de la simulación) vs. verdad inyectada:", acuerdo_plata_oro(pl_sim, V))
i0 = inc_sim.sort_values("n_alarmas", ascending=False)["incidente_id"].iloc[0]
causa = V.set_index("incidente_id")["causa_raiz_equipo"].get(i0)
html = vista_incidente(G, inc_df_sim[inc_df_sim["incidente_id"] == i0],
                       ranking=rk_sim["T1 · topológico"].query("incidente_id == @i0"), causa_real=causa, J=J,
                       titulo=f"Incidente sintético {i0}")
mostrar(html, alto=600)

> **Qué muestra.** *(Texto)* El algoritmo de etiqueta plata recupera la causa inyectada cuando las reglas son correctas: valida la lógica, no los manuales. *(Vista)* El incidente sintético más grande: el borde **dorado** marca la causa real y `#k` la posición que le dio el método topológico.

## 10. Conclusiones

- La unidad de evaluación es el **incidente** sobre el grafo físico (alarmas nuevas, ≤ 4 saltos, ≥ 3 alarmas).
- La verdad de referencia tiene dos niveles: **plata** (manuales Huawei, todos los incidentes) y **oro** (NOC, 60
  incidentes) que valida a la plata (criterio: acuerdo ≥ 0,80).
- La evaluación usa folds temporales de origen móvil, enmascaramiento y métricas Top-k, MRR, P/R/F1, equipos revisados y
  tiempo de cómputo, con IC 95 % y prueba de McNemar.
- Los métodos de referencia (T0, T0′, T1) ya están implementados; el motor del Sprint 3 (T3) se medirá con el mismo
  código.

**Pendiente:** cargar las reglas de los manuales y recibir la planilla oro del NOC.